In [13]:
using Plots
using NaturalNeighbours
using LinearAlgebra
using StableRNGs
using DelaunayTriangulation

In [ ]:
function read_custom_file(filepath::String)
    array1 = Float64[]
    array2 = Float64[]

    start_reading = false

    num_regex = r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?"

    open(filepath, "r") do file
        for line in eachline(file)

            clean_line = strip(line)

            if startswith(clean_line, "+") || startswith(clean_line, "-")
                matches = [m.match for m in eachmatch(num_regex, clean_line)]

                if length(matches) >= 2
                    push!(array1, parse(Float64, matches[1]))
                    push!(array2, parse(Float64, matches[2]))
                end
            else
                continue
            end
        end
    end
    return array1, array2
end

In [ ]:
cale = "/home/lali/TITAN-ROG-sync/julia/_Magnetism/2.S9272.PtCoCu.7/S9272-FORC-50-2000-5s"
#cale = "/home/lali/TITAN-ROG-sync/julia/_Magnetism/6.S9281.CoPt.10/S9281-FORC-50-2000-5s"
#cale = "/home/lali/TITAN-ROG-sync/julia/_Magnetism/4.S9283.CoPt.6/S9283-FORC-50-1500-5s"
H_read = Float64[]
M_read = Float64[]
H_read, M_read = read_custom_file(cale * ".txt")

In [ ]:
## Normalize
normH = 1.0e3
normM = 1.0e-3
H_read = H_read ./ normH
M_read = M_read ./ normM
H_read, M_read

In [ ]:
scatter(H_read, M_read,
    title="Scatter Plot of Data",
    xlabel="H",
    ylabel="M",
    legend=false,
    markersize=3,
    markercolor=:indianred,
    markerstrokecolor=:indianred
)

In [ ]:
## Detect FORCs (Hr, numPointsPerFORC)
Hr_read = Float64[]
numPointsPerFORC = Int[]

current_Hr = H_read[1]
push!(Hr_read, current_Hr)

counterPointsPerFORC = 1

for i in 2:(length(H_read))
    if H_read[i] < H_read[i-1]
        global current_Hr = H_read[i]
        push!(numPointsPerFORC, counterPointsPerFORC)
        global counterPointsPerFORC = 0
    end
    global counterPointsPerFORC += 1
    push!(Hr_read, current_Hr)
end
push!(numPointsPerFORC, counterPointsPerFORC)
numFORCs = length(numPointsPerFORC)

println("----------- Total $(length(numPointsPerFORC)) FORCs -----------")
println("---------  $(numPointsPerFORC[1]) first / $(numPointsPerFORC[numFORCs]) last ---------")

In [ ]:
## Save Hr-H-M original file
n = length(Hr_read)
if length(H_read) != n || length(M_read) != n
    error("Error: All three arrays must have the same length.")
end

open(cale * "_Hr-H-M_orig.dat", "w") do file
    for i in 1:n
        println(file, "$(Hr_read[i])  $(H_read[i])  $(M_read[i])")
    end
end

In [ ]:
function gimmeOneFORC(theFORC::Int64)
    contorNumPoints = 0
    H_interp = Float64[]
    M_interp = Float64[]
    #detect indexes for $(theFORC)
    if theFORC > 1
        for i in 1:theFORC-1
            contorNumPoints += numPointsPerFORC[i]
        end
    else
        contorNumPoints = 0
    end

    #myHr = Hr_read[contorNumPoints]
    startPointOnFORC = contorNumPoints + 1
    push!(H_interp, H_read[startPointOnFORC])
    push!(M_interp, M_read[startPointOnFORC])
    contorNumPoints = 1
    while (Hr_read[startPointOnFORC+contorNumPoints-1] - Hr_read[startPointOnFORC+contorNumPoints]) < 1.0e-5 #eps - compare floats
        push!(H_interp, H_read[startPointOnFORC+contorNumPoints])
        push!(M_interp, M_read[startPointOnFORC+contorNumPoints])
        contorNumPoints += 1
        if (startPointOnFORC + contorNumPoints) > length(Hr_read)
            break
        end
    end
    H_interp, M_interp
end

In [14]:
f = (x, y) -> 0.75 * exp(-((9 * x - 2)^2 + (9 * y - 2)^2) / 4) + 0.75 * exp(-(9 * x + 1)^2 / 49 - (9 * y + 1) / 10) + 0.5 * exp(-((9 * x - 7)^2 + (9 * y - 3)^2) / 4) - 0.2 * exp(-(9 * x - 4)^2 - (9 * y - 7)^2)
f′ = (x, y) -> [(exp(-(9 * x - 4)^2 - (9 * y - 7)^2) * (162 * x - 72)) / 5 - (3 * exp(-(9 * x - 2)^2 / 4 - (9 * y - 2)^2 / 4) * ((81 * x) / 2 - 9)) / 4 - (exp(-(9 * x - 7)^2 / 4 - (9 * y - 3)^2 / 4) * ((81 * x) / 2 - 63 / 2)) / 2 - (3 * exp(-(9 * y) / 10 - (9 * x + 1)^2 / 49 - 1 / 10) * ((162 * x) / 49 + 18 / 49)) / 4
    (exp(-(9 * x - 4)^2 - (9 * y - 7)^2) * (162 * y - 126)) / 5 - (3 * exp(-(9 * x - 2)^2 / 4 - (9 * y - 2)^2 / 4) * ((81 * y) / 2 - 9)) / 4 - (exp(-(9 * x - 7)^2 / 4 - (9 * y - 3)^2 / 4) * ((81 * y) / 2 - 27 / 2)) / 2 - (27 * exp(-(9 * y) / 10 - (9 * x + 1)^2 / 49 - 1 / 10)) / 40]
f′′ = (x, y) -> [(162*exp(-(9 * x - 4)^2 - (9 * y - 7)^2))/5-(243*exp(-(9 * y) / 10 - (9 * x + 1)^2 / 49 - 1 / 10))/98-(243*exp(-(9 * x - 2)^2 / 4 - (9 * y - 2)^2 / 4))/8-(81*exp(-(9 * x - 7)^2 / 4 - (9 * y - 3)^2 / 4))/4+(3*exp(-(9 * y) / 10 - (9 * x + 1)^2 / 49 - 1 / 10)*((162*x)/49+18/49)^2)/4+(3*exp(-(9 * x - 2)^2 / 4 - (9 * y - 2)^2 / 4)*((81*x)/2-9)^2)/4+(exp(-(9 * x - 7)^2 / 4 - (9 * y - 3)^2 / 4)*((81*x)/2-63/2)^2)/2-(exp(-(9 * x - 4)^2 - (9 * y - 7)^2)*(162*x-72)^2)/5 (27*exp(-(9 * y) / 10 - (9 * x + 1)^2 / 49 - 1 / 10)*((162*x)/49+18/49))/40+(3*exp(-(9 * x - 2)^2 / 4 - (9 * y - 2)^2 / 4)*((81*x)/2-9)*((81*y)/2-9))/4+(exp(-(9 * x - 7)^2 / 4 - (9 * y - 3)^2 / 4)*((81*x)/2-63/2)*((81*y)/2-27/2))/2-(exp(-(9 * x - 4)^2 - (9 * y - 7)^2)*(162*x-72)*(162*y-126))/5
    (27*exp(-(9 * y) / 10 - (9 * x + 1)^2 / 49 - 1 / 10)*((162*x)/49+18/49))/40+(3*exp(-(9 * x - 2)^2 / 4 - (9 * y - 2)^2 / 4)*((81*x)/2-9)*((81*y)/2-9))/4+(exp(-(9 * x - 7)^2 / 4 - (9 * y - 3)^2 / 4)*((81*x)/2-63/2)*((81*y)/2-27/2))/2-(exp(-(9 * x - 4)^2 - (9 * y - 7)^2)*(162*x-72)*(162*y-126))/5 (243*exp(-(9 * y) / 10 - (9 * x + 1)^2 / 49 - 1 / 10))/400+(162*exp(-(9 * x - 4)^2 - (9 * y - 7)^2))/5-(243*exp(-(9 * x - 2)^2 / 4 - (9 * y - 2)^2 / 4))/8-(81*exp(-(9 * x - 7)^2 / 4 - (9 * y - 3)^2 / 4))/4+(3*exp(-(9 * x - 2)^2 / 4 - (9 * y - 2)^2 / 4)*((81*y)/2-9)^2)/4+(exp(-(9 * x - 7)^2 / 4 - (9 * y - 3)^2 / 4)*((81*y)/2-27/2)^2)/2-(exp(-(9 * x - 4)^2 - (9 * y - 7)^2)*(162*y-126)^2)/5]

#30 (generic function with 1 method)

In [15]:
x = LinRange(0, 1, 100)
y = LinRange(0, 1, 100)
z = [f(x, y) for x in x, y in y]
∇ = [f′(x, y) for x in x, y in y]
∇₁ = first.(∇)
∇₂ = last.(∇)
H = [f′′(x, y) for x in x, y in y]
H₁₂ = getindex.(H, 2)
;

In [16]:
rng = StableRNG(9199)
x = rand(rng, 1000)
y = rand(rng, 1000)
z = f.(x, y)
tri = triangulate([x'; y'])
vorn = voronoi(tri)


Voronoi Tessellation.
    Number of generators: 1000
    Number of polygon vertices: 1982
    Number of polygons: 1000
    Weighted: false

In [17]:
points = [x'; y']
z = f.(x, y)
tri = triangulate(points)
∇g = generate_gradients(tri, z)
;

In [18]:
∇gr, _ = generate_derivatives(tri, z; method=Direct())

([(-0.0650836830700595, 0.004832653750066505), (-0.3840397795647903, -0.06102383552897363), (0.04622850856207566, -1.020017015171215), (-0.24170261810600094, -0.9297222352845911), (-2.9304791298639636, 0.2040509665601779), (-0.31251712506937085, -0.2000636772903574), (-1.7997304132682361, 0.5352919330622719), (-0.47138972468076457, -0.6514847143339548), (0.2928905688750928, 0.9808661216873321), (0.6892500942263502, -0.6464402205490298)  …  (-0.1987228929268587, -0.26142463660386694), (-0.49738621707083974, 0.21352146663765942), (-0.23618241917419025, -0.1077959561406571), (-0.9023918186289097, 1.1586363811828337), (-0.3333369901577421, 1.361700902342665), (-0.0930230048902381, -0.9076224860942793), (-1.3538433395262743, -2.9148262220627585), (2.3889789202550222, -0.9958482947983259), (-0.07110163691491153, -2.128084143216597), (2.0806511728622548, -0.11324964119053765)], [(2.148745194304252, 2.6556188986073512, 2.6587016107284476), (-1.4643482668308834, -3.680916137911643, 2.9676435797

In [19]:
#to_mat(H::NTuple{3,Float64}) = [H[1] H[3]; H[3] H[2]]

In [20]:
_, Hg = generate_derivatives(tri, z)

([(-0.0650836830700595, 0.004832653750066505), (-0.3840397795647903, -0.06102383552897363), (0.04622850856207566, -1.020017015171215), (-0.24170261810600094, -0.9297222352845911), (-2.9304791298639636, 0.2040509665601779), (-0.31251712506937085, -0.2000636772903574), (-1.7997304132682361, 0.5352919330622719), (-0.47138972468076457, -0.6514847143339548), (0.2928905688750928, 0.9808661216873321), (0.6892500942263502, -0.6464402205490298)  …  (-0.1987228929268587, -0.26142463660386694), (-0.49738621707083974, 0.21352146663765942), (-0.23618241917419025, -0.1077959561406571), (-0.9023918186289097, 1.1586363811828337), (-0.3333369901577421, 1.361700902342665), (-0.0930230048902381, -0.9076224860942793), (-1.3538433395262743, -2.9148262220627585), (2.3889789202550222, -0.9958482947983259), (-0.07110163691491153, -2.128084143216597), (2.0806511728622548, -0.11324964119053765)], [(2.148745194304252, 2.6556188986073512, 2.6587016107284476), (-1.4643482668308834, -3.680916137911643, 2.9676435797

In [21]:
myH = getindex.(Hg, 3)

#heatmap(myH)
;

In [22]:
scatter(x,y, marker_z=myH)